# Calculating Vital and Essential Workers - Walkthrough

This notebook is a guided walkthrough of the essential-worker pipeline. All the heavy
lifting lives in `src/essential_workers.py`; here we just *demonstrate* what each stage
produces, so future readers (and the test suite in `tests/test_essential_workers.py`)
can see exactly how the figures in `results/EssentialWorkersByCountry.csv` are built.

Inputs (in `data/`):

| File | Provides |
| --- | --- |
| `ISCO-08 OpinionPollCensus.xlsx` | Per-ISCO L2 essential-worker poll (Census-tagged column). |
| `Indoors_Environmentally_Controlled_data.csv` | O\*NET indoor-context %, by SOC code. |
| `ISCO_SOC_Crosswalk.csv` | SOC ↔ ISCO-08 mapping. |
| `ILO_ISCO_08_GLB.csv` | ILO employment by ISCO-08 L2, by country/year. |
| `LFData_WB_plus.xlsx` | World Bank labour-force-2024 by country. |
| `ILO_country_essential_workers_pct.xlsx` | ILO 2023 published per-country %essential. |

Outputs (written to `results/`):

* `EssentialWorkersByCountry.csv`
* `EssentialWorkersByRegion.csv`
* `Essential_Workers_Validation.csv`
* `Group_Overlap_Calibration.csv` — per-country × group overlap adjustments
* `Onsite_Housing_Worker_Requirements.csv` — essential and vital counts excl. subsistence farmers (ISCO 61+63); global + per-country

## Methodology - ILO essential workers vs our estimates of vital workers

The reference definition for essential workers we are reproducing is from:

> ILO (2023). **World Employment and Social Outlook 2023: The value of essential work.**
> International Labour Organization.
> [WCMS_871016](https://www.ilo.org/sites/default/files/wcmsp5/groups/public/@dgreports/@dcomm/@publ/documents/publication/wcms_871016.pdf)

The ILO classifies a worker as a *key worker* (essential worker) if **both** of these are true:

1. They are in a key **occupation** (ISCO-08 code listed in Table A2 of the report).
2. They are in a key **industry** (ISIC Rev.4 code listed in Table A1 of the report).

So `essential = occupation AND industry`, not just one or the other. The ILO computes its per-country shares from worker-level microdata in which every respondent has both an ISCO code and an ISIC code, so the intersection is exact.

### Our approximation

**We do not have access to ILO worker-level ISCO x ISIC microdata.** The only country-level breakdown we have is the marginal: ILO employment per ISCO-08 L2 code (`ILO_ISCO_08_GLB.csv`). To approximate the intersection we use ILO Figure A1 **global priors** (`GROUP_OVERLAP[g]`) and then **calibrate** them per country.

**Step 1 — Global priors.** For each occupational group *g*, `GROUP_OVERLAP[g]` is the globally aggregated fraction of workers in group *g* that are also in a key ISIC industry (ILO Figure A1). Armed Forces uses 0.40 (Blueprint; ILO excludes uniformed services from headline figures).

**Step 2 — Per-country calibration.** For each country with ILO ISCO employment and a published WESO %essential, one scalar `x ∈ [0, 1]` moves all eight calibratable groups together: toward 1.0 when the model under-shoots ILO, toward 0 when it over-shoots. Armed Forces overlap stays fixed at 0.40. This is implemented in `essential_workers.calibrate_group_overlaps` and logged in `results/Group_Overlap_Calibration.csv`.

**Step 3 — Countries without ILO microdata** (e.g. China): calibrated overlaps are the mean of `SIMILAR_ISO3` neighbours' calibrated values, not global Figure A1 priors.

**Vital workers** use the same calibrated overlaps (not re-fit to a separate ILO vital benchmark). `Essential_Workers_Validation.csv` reports both **model** (global overlap) and **calibrated** %essential vs ILO. Where `x` would need to exceed 1 to hit ILO (e.g. Liberia), `solver_status` is `infeasible_clipped`.

The current group overlap factors:

| Group | Overlap | Source |
| --- | ---: | --- |
| Food | 0.895 | ILO Figure A1 |
| Health | 0.819 | ILO Figure A1 |
| Retail | 0.876 | ILO Figure A1 |
| Security | 0.846 | ILO Figure A1 |
| Transport | 0.869 | ILO Figure A1 |
| Manual | 0.335 | ILO Figure A1 |
| Cleaning | 0.485 | ILO Figure A1 |
| Tech | 0.320 | ILO Figure A1 |
| Armed Forces | **0.40** | Blueprint Biosecurity's "A theory of pandemic-proof PPE" https://blueprintbiosecurity.org/u/2024/05/BB_Next-Gen-Report_PRF9-WEB-1.pdf?utm_source=bluedot-impact (as the ILO excludes uniformed services from its global figures) |

### Two further simplifications

1. **Teleworkable codes.** ILO Table A2 explicitly excludes a handful of ISCO L2 codes (13 = ICT professionals, 21 = Science/engineering, 33 = Business/admin associates, 35 = ICT technicians) on teleworkability grounds. The team's in-house "vital" poll marks some of these as vital anyway; we zero those poll-vital weights in `NON_ILO_POLL_CODES` to stay aligned with ILO's non-teleworkable scope.
2. **Subsistence farmers (ISCO 63).** ILO classes them as essential (Food group) but they work outdoors and so cannot benefit from in-room filtration. We include subsistence farmers in total counts for essential and vital workers, but remove them for the calculations of indoor essential workers and indoor vital workers.

All three simplifications above are encoded as constants at the top of `src/essential_workers.py` so they can be audited/swapped in one place.

## 0. Setup

In [1]:
import sys
from pathlib import Path

import pandas as pd

REPO = Path.cwd().parent if Path.cwd().name == 'scripts' else Path.cwd()
sys.path.insert(0, str(REPO / 'src'))

import essential_workers as ew

DATA = REPO / 'data'
RESULTS = REPO / 'results'
RESULTS.mkdir(exist_ok=True)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 220)

## 1. Quick-start: run the whole pipeline

If you just want the final numbers, this single call reproduces every CSV in `results/`
from scratch. The rest of the notebook explains each stage.

In [2]:
outputs = ew.run_pipeline(data_dir=DATA, results_dir=RESULTS, write=True)
v = outputs.validation
vm = outputs.validation_model
lf = outputs.labour_force_df
cal = outputs.overlap_calibration.detail_df

global_summary = ew.compute_global_worker_summary(lf)
global_lf = global_summary.attrs["labour_force"]

print(f"Global labour force (WB 2024): {global_lf:.3e}")
print()
print("Global worker totals (essential & vital, each split indoor vs outdoor):")
display(
    global_summary.assign(
        Workers=lambda d: d["Workers"].map(lambda x: f"{x:.3e}"),
        **{"% of Labour Force": global_summary["% of Labour Force"].map(
            lambda x: f"{x:.2f}%"
        )},
    )
)

print()
_g = outputs.onsite_housing_df.loc[outputs.onsite_housing_df["Country Code"] == "GLOBAL"].iloc[0]
print(f"On-site housing (excl. ISCO 61+63): Essential {_g['Essential Workers']:.3e}, Vital {_g['Vital Workers']:.3e}")
print(f"  -> {RESULTS / 'Onsite_Housing_Worker_Requirements.csv'}")
print(f"  -> {RESULTS / 'Group_Overlap_Calibration.csv'}")

print()
print(f"ILO validation — calibrated: global %Essential = {v.global_pct_essential:.2f}%")
print(f"  |Δ| mean: {v.mean_abs_delta_pp:.2f}pp   r: {v.correlation:.3f}   outliers >{v.outlier_threshold_pp:.0f}pp: {len(v.outlier_df)}")
print(f"ILO validation — model (global overlap): |Δ| mean = {vm.mean_abs_delta_pp:.2f}pp   r: {vm.correlation:.3f}   outliers: {len(vm.outlier_df)}")

ilo_cal = cal.loc[cal["Overlap source"] == ew.OVERLAP_SOURCE_ILO]
adj_by_group = ilo_cal.groupby("Group")["Adjustment"].apply(lambda s: s.abs().mean()).sort_values(ascending=False)
print("\nMean |calibrated − global| overlap by group (ILO-calibrated countries):")
display(adj_by_group.to_frame("mean |adjustment|"))

Global labour force (WB 2024): 3.731e+09

Global worker totals (essential & vital, each split indoor vs outdoor):


,Workers,% of Labour Force
Category,,
Essential workers,1.966e+09,52.70%
Indoor essential workers,9.031e+08,24.21%
Outdoor essential workers,1.063e+09,28.49%
Vital workers,1.278e+09,34.27%
Indoor vital workers,4.286e+08,11.49%
Outdoor vital workers,8.499e+08,22.78%



On-site housing (excl. ISCO 61+63): Essential 1.966e+09, Vital 1.278e+09
  -> /home/james/Documents/GitHub/InRoomAirFilterScaleUp/results/Onsite_Housing_Worker_Requirements.csv
  -> /home/james/Documents/GitHub/InRoomAirFilterScaleUp/results/Group_Overlap_Calibration.csv

ILO validation — calibrated: global %Essential = 52.70%
  |Δ| mean: 0.55pp   r: 0.981   outliers >10pp: 2
ILO validation — model (global overlap): |Δ| mean = 4.91pp   r: 0.883   outliers: 7

Mean |calibrated − global| overlap by group (ILO-calibrated countries):


,mean |adjustment|
Group,
Tech,0.090803
Manual,0.089749
Cleaning,0.079208
Health,0.055737
Security,0.053839
Transport,0.052223
Retail,0.051731
Food,0.050396
ArmedForces,0.000000


## 2. Stage-by-stage deep dive

### 2.1 Load the raw inputs

In [3]:
poll_df = pd.read_excel(DATA / 'ISCO-08 OpinionPollCensus.xlsx', engine='openpyxl')
onet_df = pd.read_csv(DATA / 'Indoors_Environmentally_Controlled_data.csv')
crosswalk_df = pd.read_csv(DATA / 'ISCO_SOC_Crosswalk.csv')
ilo_emp_df = pd.read_csv(DATA / 'ILO_ISCO_08_GLB.csv')
lf_df = pd.read_excel(DATA / 'LFData_WB_plus.xlsx', engine='openpyxl')
ilo_pct_df = ew.load_ilo_published_pct(DATA / 'ILO_country_essential_workers_pct.xlsx')

print('poll      :', poll_df.shape)
print('ONET ctx  :', onet_df.shape)
print('crosswalk :', crosswalk_df.shape)
print('ILO emp   :', ilo_emp_df.shape)
print('LF        :', lf_df.shape)
print('ILO pct   :', ilo_pct_df.shape)

poll      : (436, 39)
ONET ctx  : (879, 4)
crosswalk : (1125, 6)
ILO emp   : (34510, 11)
LF        : (216, 6)
ILO pct   : (90, 3)


### 2.2 Build ISCO-08 L2 weights

`build_isco_lvl2_weights` collapses the SOC-level O\*NET indoor context onto the ISCO L4
codes via the crosswalk, then averages up to the L2 codes used by ILO employment data.
It then attaches three columns whose product (with employment) determines every
downstream worker count:

| Column | Meaning | Source |
| --- | --- | --- |
| `Vital Weight POLL` | L2-averaged 0/1 from the in-house "vital" poll | poll xlsx + `NON_ILO_POLL_CODES` |
| `Context Proj` | L2-averaged indoor-environment fraction (0..1) | O\*NET via SOC -> ISCO crosswalk |
| `Essential Weight ILO` | Binary 1 if L2 code is in ILO Table A2, else 0 | `ILO_LVL2_ESSENTIAL_GROUPS` |
| `Group` / `Group Overlap` | ILO Figure A1 group + its **ISIC x ISCO overlap factor** | `ISCO_L2_TO_GROUP` + `GROUP_OVERLAP` |

The four weight columns at the right (`ISCO_08_PollWeights`, `..._Total`, `ISCO_08_ILOWeights`, `..._Total`) are just products of the four columns above. The `_Total` variants drop the indoor filter and drive the Vital / Essential totals; the non-`_Total` ones include `Context Proj` and drive Indoor Vital / Indoor Essential.

**Caveat**
*ISIC x ISCO overlap.* The `Group Overlap` column is the single per-country approximation explained in the methodology section above. The four weight columns ALL include this factor, so every count produced downstream inherits the simplification.

In [4]:
weights = ew.build_isco_lvl2_weights(poll_df, onet_df, crosswalk_df)
weights.head(10)

,Vital Weight POLL,Context Proj,Essential Weight ILO,Group,Group Overlap,ISCO_08_PollWeights,ISCO_08_ILOWeights,ISCO_08_PollWeights_Total,ISCO_08_ILOWeights_Total
ISCO-08,,,,,,,,,
01,0.400000,1.000000,1,ArmedForces,0.400,0.160000,0.400000,0.1600,0.400
02,0.400000,1.000000,1,ArmedForces,0.400,0.160000,0.400000,0.1600,0.400
03,0.400000,1.000000,1,ArmedForces,0.400,0.160000,0.400000,0.1600,0.400
11,0.000000,0.836000,0,NaN,0.000,0.000000,0.000000,0.0000,0.000
12,0.000000,0.859048,0,NaN,0.000,0.000000,0.000000,0.0000,0.000
13,0.000000,0.783500,0,NaN,0.000,0.000000,0.000000,0.0000,0.000
14,0.000000,0.840000,0,NaN,0.000,0.000000,0.000000,0.0000,0.000
21,0.000000,0.853318,0,NaN,0.000,0.000000,0.000000,0.0000,0.000
22,0.533333,0.897970,1,Health,0.819,0.392233,0.735437,0.4368,0.819


In [5]:
weights[[
    'Vital Weight POLL', 'Context Proj', 'Essential Weight ILO',
    'Group', 'Group Overlap',
    'ISCO_08_PollWeights', 'ISCO_08_PollWeights_Total',
    'ISCO_08_ILOWeights', 'ISCO_08_ILOWeights_Total',
]].head(15)

,Vital Weight POLL,Context Proj,Essential Weight ILO,Group,Group Overlap,ISCO_08_PollWeights,ISCO_08_PollWeights_Total,ISCO_08_ILOWeights,ISCO_08_ILOWeights_Total
ISCO-08,,,,,,,,,
01,0.400000,1.000000,1,ArmedForces,0.400,0.160000,0.160000,0.400000,0.400
02,0.400000,1.000000,1,ArmedForces,0.400,0.160000,0.160000,0.400000,0.400
03,0.400000,1.000000,1,ArmedForces,0.400,0.160000,0.160000,0.400000,0.400
11,0.000000,0.836000,0,NaN,0.000,0.000000,0.000000,0.000000,0.000
12,0.000000,0.859048,0,NaN,0.000,0.000000,0.000000,0.000000,0.000
13,0.000000,0.783500,0,NaN,0.000,0.000000,0.000000,0.000000,0.000
14,0.000000,0.840000,0,NaN,0.000,0.000000,0.000000,0.000000,0.000
21,0.000000,0.853318,0,NaN,0.000,0.000000,0.000000,0.000000,0.000
22,0.533333,0.897970,1,Health,0.819,0.392233,0.436800,0.735437,0.819


### 2.3 Extract per-country ILO employment by ISCO L2

`build_employment_by_isco` picks the most recent non-NaN year for every (country,
ISCO-L2) cell and converts thousands-of-workers to absolute counts.

In [6]:
employment = ew.build_employment_by_isco(ilo_emp_df)
sample_country = next(iter(employment))
print(f'Countries with ILO ISCO L2 employment: {len(employment)}')
print(f'\nExample: {sample_country!r}')
pd.Series(employment[sample_country]).head(15)

Countries with ILO ISCO L2 employment: 147

Example: 'Afghanistan'


Tot    7679474.0
02      209728.0
11        7876.0
12       89945.0
13       23102.0
14       10688.0
21       22794.0
22       63733.0
23      236394.0
24        9108.0
26       58692.0
32        9930.0
33       76513.0
34        7447.0
41        2385.0
dtype: float64

### 2.4 Compute the per-country worker dictionaries

`compute_worker_dicts` produces seven country-keyed dicts: indoor-vital, vital,
indoor-essential, essential, plus the armed-forces subtotals.

In [7]:
workers = ew.compute_worker_dicts(employment, weights)
pd.DataFrame({
    'Indoor Vital Workers':     workers.ivw_poll,
    'Vital Workers':            workers.vw_poll,
    'Indoor Essential Workers': workers.iew_ilo,
    'Essential Workers':        workers.ew_ilo,
    '%Indoor Vital':            workers.ivw_pc,
    '%Vital':                   workers.vw_pc,
    '%Indoor Essential':        workers.iew_pc,
    '%Essential':               workers.ew_pc,
    'Armed Forces Indoor Essential':      workers.af_indoor_essential,
    'Armed Forces Essential':             workers.af_essential,
}).head(10)

,Indoor Vital Workers,Vital Workers,Indoor Essential Workers,Essential Workers,%Indoor Vital,%Vital,%Indoor Essential,%Essential,Armed Forces Indoor Essential,Armed Forces Essential
Afghanistan,7.058073e+05,4.074131e+06,1.568858e+06,5276537.227,0.091908,0.530522,0.204292,0.687096,148277.6,148277.6
Angola,7.328789e+05,6.264542e+06,2.522504e+06,8646941.187,0.062216,0.531812,0.214141,0.734060,55102.0,55102.0
Albania,1.613663e+05,5.045112e+05,3.197399e+05,731975.343,0.121860,0.380994,0.241460,0.552770,2207.2,2207.2
United Arab Emirates,5.123877e+05,1.064419e+06,1.361435e+06,2447013.427,0.070161,0.145750,0.186421,0.335069,17615.2,17615.2
Argentina,9.902605e+05,1.817993e+06,3.344126e+06,5172564.261,0.074258,0.136329,0.250771,0.387883,0.0,0.0
Australia,1.212061e+06,2.146447e+06,2.966865e+06,4599613.521,0.084043,0.148831,0.205718,0.318930,2270.4,2270.4
Austria,3.819510e+05,6.755791e+05,9.347599e+05,1451757.152,0.085200,0.150698,0.208512,0.323835,3989.6,3989.6
Burundi,1.067606e+05,3.866881e+06,2.877007e+05,4129835.716,0.021552,0.780611,0.058078,0.833694,579.6,579.6
Belgium,3.863393e+05,6.579583e+05,9.405924e+05,1467496.478,0.076831,0.130847,0.187054,0.291839,7538.8,7538.8
Benin,7.309558e+05,2.455399e+06,1.841174e+06,3893634.268,0.134243,0.450942,0.338138,0.715079,2940.8,2940.8


### 2.5 Attach to the labour-force table and back-fill neighbours

Countries with no ILO ISCO L2 data have their %-columns filled in by averaging
geographically/economically similar neighbours (see
`essential_workers.SIMILAR_ISO3`). The back-fill iterates until stable.

In [8]:
lf = ew.prepare_labour_force(lf_df)
lf = ew.attach_pct_columns(lf, workers)
print(f'Rows with NaN before back-fill: {lf["%Essential Workers"].isna().sum()}')

lf = ew.backfill_neighbours(lf)
print(f'Rows with NaN after  back-fill: {lf["%Essential Workers"].isna().sum()}')
lf.head(10)

Rows with NaN before back-fill: 74
Rows with NaN after  back-fill: 0


,Country Name,Country Code,Indicator Name,Labour Force (2024),Unnamed: 4,All unreferenced entries are sourced from Worldbank: https://data.worldbank.org/indicator/SL.TLF.TOTL.IN,Region,%Indoor Essential Workers,%Indoor Vital Workers,%Essential Workers,%Vital Workers,%Armed Forces (Indoor Essential),%Armed Forces (Essential)
0,Aruba,ABW,"Labor force, total",55826.0,NaN,https://cbs.aw/wp/index.php/2025/06/06/labor-f...,Caribbean,0.226552,0.101885,0.420866,0.239804,NaN,NaN
1,Afghanistan,AFG,"Labor force, total",9130000.0,NaN,NaN,Southern Asia,0.204292,0.091908,0.687096,0.530522,0.019308,0.019308
2,Angola,AGO,"Labor force, total",16000000.0,NaN,NaN,Middle Africa,0.214141,0.062216,0.734060,0.531812,0.004678,0.004678
3,Albania,ALB,"Labor force, total",1370000.0,NaN,NaN,Southern Europe,0.241460,0.121860,0.552770,0.380994,0.001667,0.001667
4,Andorra,AND,"Labor force, total",50504.0,NaN,https://www.andorra-solutions.com/blog/2023/10...,Southern Europe,0.201871,0.067127,0.317204,0.122881,NaN,NaN
5,United Arab Emirates,ARE,"Labor force, total",7090000.0,NaN,NaN,Western Asia,0.186421,0.070161,0.335069,0.145750,0.002412,0.002412
6,Argentina,ARG,"Labor force, total",22300000.0,NaN,NaN,South America,0.250771,0.074258,0.387883,0.136329,0.000000,0.000000
7,Armenia,ARM,"Labor force, total",1510000.0,NaN,NaN,Western Asia,0.216336,0.111381,0.570992,0.421732,NaN,NaN
8,American Samoa,ASM,"Labor force, total",55958.0,NaN,https://www.ilo.org/media/533476/download,Polynesia,0.212002,0.093378,0.554787,0.387975,NaN,NaN
9,Antigua and Barbuda,ATG,"Labor force, total",46540.0,2023,https://m.elibrary.imf.org/view/journals/002/2...,Caribbean,0.201787,0.139491,0.757144,0.664737,NaN,NaN


### 2.6 Multiply percentages by labour force and aggregate regionally

In [9]:
lf = ew.compute_absolute_counts(lf)
regional = ew.aggregate_by_region(lf)

global_summary = ew.compute_global_worker_summary(lf)
print(f"Global labour force: {global_summary.attrs['labour_force']:.3e}")
display(global_summary)

regional

Global labour force: 3.731e+09


,Workers,% of Labour Force
Category,,
Essential workers,1.946191e+09,52.163930
Indoor essential workers,8.939930e+08,23.961777
Outdoor essential workers,1.052198e+09,28.202153
Vital workers,1.272051e+09,34.094901
Indoor vital workers,4.264385e+08,11.429870
Outdoor vital workers,8.456125e+08,22.665031


,Region,Labour Force (2024),Indoor Essential Workers,Indoor Vital Workers,Essential Workers,Vital Workers,Armed Forces (Indoor Essential),Armed Forces (Essential),%Indoor Essential Workers,%Indoor Vital Workers,%Essential Workers,%Vital Workers,%Armed Forces (Indoor Essential),%Armed Forces (Essential)
0,Australia and New Zealand,1.802000e+07,3.707038e+06,1.514446e+06,5.747124e+06,2.681942e+06,2345.649214,2345.649214,0.205718,0.084043,0.318930,0.148831,0.000130,0.000130
1,Caribbean,1.989858e+07,4.585432e+06,1.632874e+06,8.288341e+06,4.018966e+06,17050.518855,17050.518855,0.230440,0.082060,0.416529,0.201973,0.000857,0.000857
2,Central America,8.375000e+07,2.264474e+07,7.431684e+06,4.165108e+07,1.980532e+07,48469.372257,48469.372257,0.270385,0.088737,0.497326,0.236481,0.000579,0.000579
3,Central Asia,3.272000e+07,6.921162e+06,3.448790e+06,1.849502e+07,1.338426e+07,5719.945979,5719.945979,0.211527,0.105403,0.565251,0.409054,0.000175,0.000175
4,Eastern Africa,2.116690e+08,4.180339e+07,2.296756e+07,1.520655e+08,1.237637e+08,205581.981220,205581.981220,0.197494,0.108507,0.718412,0.584704,0.000971,0.000971
5,Eastern Asia,8.963720e+08,2.160692e+08,9.978138e+07,4.442444e+08,2.767872e+08,3146.976207,3146.976207,0.241049,0.111317,0.495603,0.308786,0.000004,0.000004
6,Eastern Europe,1.465706e+08,3.080778e+07,1.437134e+07,5.481879e+07,3.065141e+07,113411.587704,113411.587704,0.210191,0.098051,0.374009,0.209124,0.000774,0.000774
7,Melanesia,4.733000e+06,1.334485e+06,5.522798e+05,3.136924e+06,2.048281e+06,2781.961296,2781.961296,0.281953,0.116687,0.662777,0.432766,0.000588,0.000588
8,Micronesia,1.778560e+05,3.969716e+04,1.654360e+04,8.206178e+04,4.852754e+04,61.723348,61.723348,0.223198,0.093017,0.461394,0.272847,0.000347,0.000347
9,Middle Africa,7.833350e+07,1.575627e+07,8.283915e+06,5.374187e+07,4.365786e+07,80173.274738,80173.274738,0.201143,0.105752,0.686065,0.557333,0.001023,0.001023


### 2.7 Sense-check against ILO 2023 published figures

`validate_against_ilo` cross-references our per-country %Essential against the ILO's
published "Share of key workers" table. The two sense checks the repo cares about are:

1. **Global** %Essential should be within 5pp of the ILO global figure (52%).
2. **Per-country** Δ vs ILO should be ≤10pp (currently fails for 7 countries; see
   `tests/test_essential_workers.py::test_no_country_deviates_more_than_10pp`).

The ILO figures are computed from worker-level ISCO x ISIC microdata, so they are the *true* intersection of essential occupation and essential industry. Our pipeline approximates that intersection with the single global `GROUP_OVERLAP` factor per occupational group described in the methodology section. The per-country deviations below are therefore mostly the cost of that simplification (worst in agrarian low-income economies, where the true "Manual" overlap is much higher than the global 0.335 we use), not bugs in either calculation.

In [10]:
validation = ew.validate_against_ilo(lf, ilo_pct_df)
print(f'Global %Essential:   {validation.global_pct_essential:.2f}%')
print(f'Mean |Δ| (pp):       {validation.mean_abs_delta_pp:.2f}')
print(f'Pearson r:           {validation.correlation:.3f}')
print()
print(f'Per-country outliers above {validation.outlier_threshold_pp:.0f}pp:')
validation.outlier_df[['Country Name', 'Our %Essential (pct)', 'ILO %essential (published)', 'Delta (pp)']]

Global %Essential:   52.16%
Mean |Δ| (pp):       4.91
Pearson r:           0.883

Per-country outliers above 10pp:


,Country Name,Our %Essential (pct),ILO %essential (published),Delta (pp)
44,Liberia,40.292200,68.48,-28.187800
79,Tuvalu,29.664249,55.37,-25.705751
25,"Micronesia, Fed. Sts.",43.256440,62.74,-19.483560
42,Laos,75.726566,56.96,18.766566
56,Nigeria,46.669494,63.05,-16.380506
49,Mexico,47.548079,58.69,-11.141921
47,Madagascar,75.714402,86.82,-11.105598


In [11]:
# show the global totals for each of the seven categories


## 3. Re-run the test suite

Run the fast subset with `pytest`, and the slow ILO sense-checks with `pytest --full-data`.